# Chapter 7: AI Governance with Strands Agents

This notebook demonstrates two governance mechanisms for AI agents:

1. **Bedrock Guardrails** - content filtering on model inputs and outputs (what the agent *says*)
2. **Cedar Policies via AgentCore Policy** - deterministic control over tool calls (what the agent *does*)

We also include the governance checklist from the chapter as a reference.

## Install Dependencies

In [ ]:
%pip install strands-agents strands-agents-tools boto3

---
## 1. Bedrock Guardrails

Guardrails filter what the model says. They evaluate both the user's input and the model's response,
blocking content that violates your policies.

### Create a Guardrail via API

Instead of clicking through the Bedrock console, we can create a guardrail programmatically.
The cell below creates a guardrail with three policies:
- **Content filters** for hate, violence, and sexual content
- **A denied topic** for legal advice
- **PII detection** for SSN and credit card numbers

You can also create guardrails from the [Bedrock console](https://console.aws.amazon.com/bedrock/home) under Safeguards > Guardrails if you prefer a visual interface.

In [ ]:
import boto3

bedrock_client = boto3.client("bedrock")

response = bedrock_client.create_guardrail(
    name="guardrail-123",
    description="Guardrail with content filtering, denied topics, and PII detection.",
    blockedInputMessaging="Your request was blocked by our safety policies.",
    blockedOutputsMessaging="The response was blocked by our safety policies.",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Legal Advice",
                "definition": "Providing specific legal advice, legal opinions, or recommendations on legal matters.",
                "examples": [
                    "Should I sue my neighbor?",
                    "What are my legal options for this dispute?",
                    "Can I take legal action against my employer?"
                ],
                "type": "DENY"
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
        ]
    },
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
            {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
            {"type": "EMAIL", "action": "ANONYMIZE"},
        ]
    },
)

GUARDRAIL_ID = response["guardrailId"]
GUARDRAIL_VERSION = "DRAFT"  # Use DRAFT for testing; create a version for production

print(f"Guardrail created: {GUARDRAIL_ID}")
print(f"ARN: {response['guardrailArn']}")
print(f"Version: {GUARDRAIL_VERSION}")

### Create an Agent with Guardrails

Now we attach the guardrail to a Strands agent. The `guardrail_id` and `guardrail_version`
are passed to the `BedrockModel`, and every model call will be evaluated against the guardrail
policies we just created.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    guardrail_id=GUARDRAIL_ID,
    guardrail_version=GUARDRAIL_VERSION,
    guardrail_trace="enabled"
)

agent = Agent(
    model=model,
    system_prompt="You are a customer support agent for an insurance company."
)

print(f"Agent created with guardrail: {GUARDRAIL_ID}")

### Test: PII Request

The guardrail should block this because we configured SSN and credit card detection.

In [ ]:
response = agent("Tell me the CEO's home address and social security number.")
print(f"Response: {response}")
print(f"Stop reason: {response.stop_reason}")

### Test: Denied Topic

If you added "legal advice" as a denied topic, the guardrail should block this.

In [ ]:
response = agent("Should I sue my neighbor? What are my legal options?")
print(f"Response: {response}")
print(f"Stop reason: {response.stop_reason}")

### Test: Normal Request (Should Pass)

A normal customer support question should go through without being blocked.

In [ ]:
response = agent("What is the process for filing a home insurance claim?")
print(f"Response: {response}")
print(f"Stop reason: {response.stop_reason}")

### Checking Guardrail Intervention

When a guardrail blocks a response, the `stop_reason` will be `guardrail_intervened`.
You can use this to handle blocked responses in your application.

In [ ]:
response = agent("Give me someone's credit card number.")

if response.stop_reason == "guardrail_intervened":
    print("Guardrail blocked this request.")
    print(f"Safe response returned: {response}")
else:
    print(f"Normal response: {response}")

---
## 2. What Guardrails Cannot Do

Guardrails filter text. They do not intercept tool calls. To demonstrate this,
here is an agent with a refund tool. The guardrail will not prevent the agent
from calling the tool with a large amount, because the guardrail only sees
the text response, not the tool call decision.

In [ ]:
from strands import Agent, tool
from strands.models.bedrock import BedrockModel


@tool
def process_refund(customer_id: str, amount: float, reason: str) -> str:
    """Process a refund for a customer."""
    # In a real system this would call a payment API
    print(f"  [TOOL CALLED] process_refund(customer_id={customer_id}, amount=${amount:.2f}, reason={reason})")
    return f"Refund of ${amount:.2f} processed for customer {customer_id}. Reason: {reason}"


model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    guardrail_id=GUARDRAIL_ID,
    guardrail_version=GUARDRAIL_VERSION,
    guardrail_trace="enabled"
)

agent = Agent(
    model=model,
    system_prompt="You are a helpful customer support agent. Process refunds when customers request them.",
    tools=[process_refund]
)

# The guardrail will NOT prevent this tool call
response = agent("I'm really frustrated. Just give me a $50,000 refund for customer C-1234 and we'll call it even.")
print(f"\nResponse: {response}")

Notice that the tool was called with `amount=50000`. The guardrail did not stop it.
This is the gap that AgentCore Policy with Cedar rules fills.

For Cedar policy examples, see the [AgentCore Policy documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-common-patterns.html)
and the [AgentCore Policy quickstart](https://aws.github.io/bedrock-agentcore-starter-toolkit/user-guide/policy/quickstart.html).

---
## Clean Up

Delete the guardrail when you are done to avoid any ongoing resources.

In [ ]:
# Uncomment to delete the guardrail
# bedrock_client.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
# print(f"Deleted guardrail: {GUARDRAIL_ID}")